# Assignment 5 — Histogram Equalization and Histogram Matching

**Objective 1:** Enhance the contrast of an input grayscale image using histogram equalization and display the original and equalized images.

**Objective 2:** Perform histogram matching of an input grayscale image with a reference image.

### How to use
1. Run the setup/import cell.
2. Run the upload cell and select the input image. For Question 2, also select a reference image.
3. Run the remaining cells in order.

The notebook works in Jupyter Notebook, JupyterLab, and Google Colab. Images can be JPG, JPEG, PNG, or BMP.

In [ ]:
# Install required libraries (run once if they are not already installed)
%pip install -q numpy matplotlib pillow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

print("Libraries imported successfully.")

## Upload the images

For Question 1, upload one input image.

For Question 2, upload both:
- **Input image:** the image whose histogram will be changed.
- **Reference image:** the image whose intensity distribution should be matched.

The code below supports both Google Colab and ordinary Jupyter environments.

In [ ]:
# Upload images
try:
    from google.colab import files
    uploaded = files.upload()
    image_paths = list(uploaded.keys())
except ImportError:
    import tkinter as tk
    from tkinter.filedialog import askopenfilenames

    root = tk.Tk()
    root.withdraw()
    selected = askopenfilenames(
        title="Select input image and reference image",
        filetypes=[("Image files", "*.png *.jpg *.jpeg *.bmp *.tif *.tiff")]
    )
    image_paths = list(selected)

if len(image_paths) < 1:
    raise ValueError("No image selected. Please run this cell again and select an image.")

print("Selected files:")
for i, path in enumerate(image_paths, start=1):
    print(f"{i}. {path}")

if len(image_paths) < 2:
    print("\nFor Question 2, run this cell again and select the reference image as well.")

In [ ]:
def load_grayscale_image(path):
    """Open an image and convert it to 8-bit grayscale (pixel values 0–255)."""
    image = Image.open(path).convert("L")
    return np.array(image, dtype=np.uint8)


# Load the first selected image as the input image
input_image = load_grayscale_image(image_paths[0])

plt.figure(figsize=(6, 5))
plt.imshow(input_image, cmap="gray", vmin=0, vmax=255)
plt.title("Original Input Grayscale Image")
plt.axis("off")
plt.show()

print("Input image shape:", input_image.shape)
print("Minimum pixel value:", input_image.min())
print("Maximum pixel value:", input_image.max())

## Question 1 — Histogram Equalization

Histogram equalization improves image contrast by redistributing pixel intensity values.

### Algorithm
1. Calculate the histogram of the grayscale image.
2. Calculate the cumulative distribution function (CDF).
3. Normalize the CDF to the range 0–255.
4. Map every original pixel value to its new intensity using the normalized CDF.
5. Display the original and equalized images.

In [ ]:
def histogram_equalization(image):
    """Perform histogram equalization on an 8-bit grayscale image."""
    # Calculate histogram for all 256 possible intensity values
    histogram = np.bincount(image.ravel(), minlength=256)

    # Calculate cumulative distribution function (CDF)
    cdf = histogram.cumsum()

    # Ignore empty intensity levels when finding the first non-zero CDF
    nonzero_cdf = cdf[cdf > 0]
    cdf_min = nonzero_cdf[0]

    # Normalize CDF to the range 0–255
    total_pixels = image.size
    denominator = total_pixels - cdf_min

    if denominator == 0:
        # A constant image cannot be contrast-stretched by equalization
        return image.copy()

    lookup_table = np.round(
        (cdf - cdf_min) / denominator * 255
    ).clip(0, 255).astype(np.uint8)

    # Replace every pixel with its mapped intensity
    equalized_image = lookup_table[image]
    return equalized_image


equalized_image = histogram_equalization(input_image)

# Display original and equalized images side by side
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap="gray", vmin=0, vmax=255)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(equalized_image, cmap="gray", vmin=0, vmax=255)
plt.title("Histogram Equalized Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compare the histograms before and after equalization
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(input_image.ravel(), bins=256, range=(0, 256), color="gray")
plt.title("Histogram of Original Image")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.xlim(0, 255)

plt.subplot(1, 2, 2)
plt.hist(equalized_image.ravel(), bins=256, range=(0, 256), color="gray")
plt.title("Histogram of Equalized Image")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.xlim(0, 255)

plt.tight_layout()
plt.show()

In [ ]:
# Save the equalized output image
Image.fromarray(equalized_image).save("equalized_output.png")
print("Saved: equalized_output.png")

## Question 2 — Histogram Matching

Histogram matching (also called histogram specification) changes the intensity distribution of the input image so that it resembles the distribution of a reference image.

### Algorithm
1. Calculate the histogram and CDF of the input image.
2. Calculate the histogram and CDF of the reference image.
3. For every input intensity, find the reference intensity whose CDF value is closest.
4. Use this mapping to transform the input image.
5. Display the input, reference, and matched images.

In [ ]:
# Select the reference image
if len(image_paths) >= 2:
    reference_path = image_paths[1]
else:
    try:
        from google.colab import files
        print("Upload the reference image:")
        uploaded_reference = files.upload()
        if not uploaded_reference:
            raise ValueError("No reference image uploaded.")
        reference_path = list(uploaded_reference.keys())[0]
    except ImportError:
        root = tk.Tk()
        root.withdraw()
        selected_reference = askopenfilenames(
            title="Select reference image",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.bmp *.tif *.tiff")]
        )
        if not selected_reference:
            raise ValueError("No reference image selected.")
        reference_path = selected_reference[0]

reference_image = load_grayscale_image(reference_path)

plt.figure(figsize=(6, 5))
plt.imshow(reference_image, cmap="gray", vmin=0, vmax=255)
plt.title("Reference Grayscale Image")
plt.axis("off")
plt.show()

print("Reference image shape:", reference_image.shape)

In [ ]:
def histogram_matching(input_img, reference_img):
    """Match the grayscale histogram of input_img to reference_img."""
    # Calculate normalized histograms (probability distributions)
    input_hist = np.bincount(input_img.ravel(), minlength=256).astype(np.float64)
    reference_hist = np.bincount(reference_img.ravel(), minlength=256).astype(np.float64)

    input_hist /= input_img.size
    reference_hist /= reference_img.size

    # Calculate cumulative distribution functions
    input_cdf = np.cumsum(input_hist)
    reference_cdf = np.cumsum(reference_hist)

    # Build a lookup table: each input intensity maps to the closest
    # reference intensity having a similar cumulative probability.
    lookup_table = np.zeros(256, dtype=np.uint8)

    for intensity in range(256):
        closest_reference_intensity = np.argmin(
            np.abs(reference_cdf - input_cdf[intensity])
        )
        lookup_table[intensity] = closest_reference_intensity

    # Apply the mapping to every pixel in the input image
    matched_img = lookup_table[input_img]
    return matched_img


matched_image = histogram_matching(input_image, reference_image)

# Display input, reference, and histogram-matched images
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(input_image, cmap="gray", vmin=0, vmax=255)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(reference_image, cmap="gray", vmin=0, vmax=255)
plt.title("Reference Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(matched_image, cmap="gray", vmin=0, vmax=255)
plt.title("Histogram Matched Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compare the input, reference, and matched histograms
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.hist(input_image.ravel(), bins=256, range=(0, 256), color="gray")
plt.title("Input Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.xlim(0, 255)

plt.subplot(1, 3, 2)
plt.hist(reference_image.ravel(), bins=256, range=(0, 256), color="gray")
plt.title("Reference Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.xlim(0, 255)

plt.subplot(1, 3, 3)
plt.hist(matched_image.ravel(), bins=256, range=(0, 256), color="gray")
plt.title("Matched Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.xlim(0, 255)

plt.tight_layout()
plt.show()

In [ ]:
# Save the histogram-matched output image
Image.fromarray(matched_image).save("histogram_matched_output.png")
print("Saved: histogram_matched_output.png")

## Result

**Question 1:** The input grayscale image is enhanced using histogram equalization. The original image, equalized image, and their histograms are displayed.

**Question 2:** The input image is transformed using the cumulative distribution functions of the input and reference images. The input, reference, matched image, and their histograms are displayed.

### Important viva points
- **Histogram:** A graph showing the frequency of each pixel intensity.
- **Grayscale image:** An image whose pixel values represent brightness, usually from 0 (black) to 255 (white).
- **Histogram equalization:** A contrast-enhancement technique based on the cumulative distribution function.
- **CDF:** The cumulative sum of the histogram or probability distribution.
- **Histogram matching:** A technique that transforms an image so its intensity distribution approximates that of a reference image.
- **Difference:** Equalization redistributes intensities to improve contrast; matching uses a reference image's intensity distribution.